In [15]:
from langchain_community.vectorstores import FAISS
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_core.documents import Document
from langchain_huggingface import ChatHuggingFace,HuggingFaceEndpoint
from langchain_classic.retrievers.multi_query import MultiQueryRetriever
from dotenv import load_dotenv

In [2]:
all_docs = [
    Document(page_content="Regular walking boosts heart health and can reduce symptoms of depression.", metadata={"source": "H1"}),
    Document(page_content="Consuming leafy greens and fruits helps detox the body and improve longevity.", metadata={"source": "H2"}),
    Document(page_content="Deep sleep is crucial for cellular repair and emotional regulation.", metadata={"source": "H3"}),
    Document(page_content="Mindfulness and controlled breathing lower cortisol and improve mental clarity.", metadata={"source": "H4"}),
    Document(page_content="Drinking sufficient water throughout the day helps maintain metabolism and energy.", metadata={"source": "H5"}),
    Document(page_content="The solar energy system in modern homes helps balance electricity demand.", metadata={"source": "I1"}),
    Document(page_content="Python balances readability with power, making it a popular system design language.", metadata={"source": "I2"}),
    Document(page_content="Photosynthesis enables plants to produce energy by converting sunlight.", metadata={"source": "I3"}),
    Document(page_content="The 2022 FIFA World Cup was held in Qatar and drew global energy and excitement.", metadata={"source": "I4"}),
    Document(page_content="Black holes bend spacetime and store immense gravitational energy.", metadata={"source": "I5"}),
]

In [5]:
embedding=HuggingFaceEmbeddings(model_name='sentence-transformers/all-MiniLM-L6-v2')

vectorestore=FAISS.from_documents(documents=all_docs,embedding=embedding)

Loading weights: 100%|██████████| 103/103 [00:00<00:00, 8209.28it/s]
BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


In [6]:
similarity_retriever=vectorestore.as_retriever(search_type="similarity",search_kwargs={"k":5})

In [16]:
load_dotenv()

llm=HuggingFaceEndpoint(
    repo_id="openai/gpt-oss-120b",
    task="text-generation"
)

model=ChatHuggingFace(llm=llm)

multiquery_retriever=MultiQueryRetriever.from_llm(
    retriever=vectorestore.as_retriever(search_kwargs={"k":5}),
    llm=model
)

In [17]:
query="How to improve energy levels and maintain balance"

In [18]:
similarity_results=similarity_retriever.invoke(query)
multiquery_results=multiquery_retriever.invoke(query)


In [19]:
print(similarity_results)

[Document(id='b87fdd91-8c18-4677-9464-862c752dbf8c', metadata={'source': 'H5'}, page_content='Drinking sufficient water throughout the day helps maintain metabolism and energy.'), Document(id='1e823974-7c52-41cf-b532-2cd6bd4a0c5f', metadata={'source': 'I1'}, page_content='The solar energy system in modern homes helps balance electricity demand.'), Document(id='71bb2c61-c4ed-4339-bad2-129a0ad533a4', metadata={'source': 'H2'}, page_content='Consuming leafy greens and fruits helps detox the body and improve longevity.'), Document(id='4ebf2e8b-576c-4e78-aafe-d1862946191a', metadata={'source': 'H4'}, page_content='Mindfulness and controlled breathing lower cortisol and improve mental clarity.'), Document(id='9f0f008f-7a8f-405d-86d7-22ec5835037b', metadata={'source': 'I3'}, page_content='Photosynthesis enables plants to produce energy by converting sunlight.')]


In [20]:
print(multiquery_results)

[Document(id='b87fdd91-8c18-4677-9464-862c752dbf8c', metadata={'source': 'H5'}, page_content='Drinking sufficient water throughout the day helps maintain metabolism and energy.'), Document(id='4ebf2e8b-576c-4e78-aafe-d1862946191a', metadata={'source': 'H4'}, page_content='Mindfulness and controlled breathing lower cortisol and improve mental clarity.'), Document(id='ea40e73e-2c72-43f6-9a2c-195cafd3c2bb', metadata={'source': 'H1'}, page_content='Regular walking boosts heart health and can reduce symptoms of depression.'), Document(id='71bb2c61-c4ed-4339-bad2-129a0ad533a4', metadata={'source': 'H2'}, page_content='Consuming leafy greens and fruits helps detox the body and improve longevity.'), Document(id='1e823974-7c52-41cf-b532-2cd6bd4a0c5f', metadata={'source': 'I1'}, page_content='The solar energy system in modern homes helps balance electricity demand.'), Document(id='f31993a7-6861-405f-becd-3ce9955b164c', metadata={'source': 'H3'}, page_content='Deep sleep is crucial for cellular r

In [21]:
for i, doc in enumerate(similarity_results):
    print(f"\n--- Result {i+1} ---")
    print(doc.page_content)

print("*"*150)

for i, doc in enumerate(multiquery_results):
    print(f"\n--- Result {i+1} ---")
    print(doc.page_content)


--- Result 1 ---
Drinking sufficient water throughout the day helps maintain metabolism and energy.

--- Result 2 ---
The solar energy system in modern homes helps balance electricity demand.

--- Result 3 ---
Consuming leafy greens and fruits helps detox the body and improve longevity.

--- Result 4 ---
Mindfulness and controlled breathing lower cortisol and improve mental clarity.

--- Result 5 ---
Photosynthesis enables plants to produce energy by converting sunlight.
******************************************************************************************************************************************************

--- Result 1 ---
Drinking sufficient water throughout the day helps maintain metabolism and energy.

--- Result 2 ---
Mindfulness and controlled breathing lower cortisol and improve mental clarity.

--- Result 3 ---
Regular walking boosts heart health and can reduce symptoms of depression.

--- Result 4 ---
Consuming leafy greens and fruits helps detox the body and imp